# Feature Engineering

**Input:** `data/processed/fasting_subsample.pkl` — output of notebook 01.  
**Output:** `data/processed/analysis_ready.pkl` + `data/processed/feature_sets.json`

The dataset contains **multiple rows per participant** (one per accelerometer day). This notebook aggregates accelerometer data to participant level, engineers biological features, merges everything, and defines the three feature sets used for modelling.

### Notebook outline

| Step | Description |
|------|-------------|
| 1. Load | Read fasting subsample |
| 2. Outcomes | Define hard CVD (primary) and composite CVD (sensitivity) |
| 3. Accelerometer | Quality filter days; aggregate to participant level |
| 4. Biological features | Participant-level biomarkers + derived clinical features |
| 5. Merge | Combine into one analysis-ready row per participant |
| 6. Feature sets | Define biological, digital, and combined feature lists |
| 7. Missingness | Audit and impute |
| 8. Save | Write pickle and feature set definitions |

In [1]:
%cd ~/Documents/biomarker-actionability

/Users/mkopy/Documents/biomarker-actionability


In [2]:
import pandas as pd
import numpy as np
import json
import os

pd.options.future.infer_string = False  # use object dtype for strings (pandas 3.0 compat)

from src.features import (
    aggregate_accelerometer,
    build_biological_features,
    BIOLOGICAL_FEATURES,
    DIGITAL_FEATURES,
    COMBINED_FEATURES,
    TARGETS,
)

## 1. Load Fasting Subsample

In [3]:
df = pd.read_pickle('data/processed/fasting_subsample.pkl')

n_rows = len(df)
n_participants = df['participant_id'].nunique()
rows_per_participant = n_rows / n_participants

print(f"Rows            : {n_rows:,}")
print(f"Participants    : {n_participants:,}")
print(f"Avg rows/person : {rows_per_participant:.1f}  (accelerometer days)")
print(f"Columns         : {df.shape[1]}")

Rows            : 115,700
Participants    : 6,205
Avg rows/person : 18.6  (accelerometer days)
Columns         : 255


## 2. Outcome Variables

- **Hard CVD (primary):** actual diagnosed events — MI, stroke, or coronary heart disease.  
- **Composite CVD (sensitivity):** hard events plus statin use as a high-risk proxy.

At 18.2% prevalence, hard CVD sits in the workable range for decision curve analysis. The composite (65.3%) is dominated by statin use (77% of the subsample) and is kept only as a sensitivity check — see notebook 01 Section 6 for the full rationale.

In [4]:
# Hard CVD: actual diagnosed cardiovascular events
df['cvd_hard'] = (
    (df['ever_told_heart_attack'] == 1) |
    (df['ever_told_stroke'] == 1) |
    (df['ever_told_coronary_heart_disease'] == 1)
).astype(int)

# Composite: hard events + statin use (taking_cholesterol_medication, NHANES BPQ070)
df['cvd_composite'] = (
    (df['cvd_hard'] == 1) |
    (df['taking_cholesterol_medication'] == 1)
).astype(int)

# Outcomes are constant per participant — confirm no within-person variation
assert df.groupby('participant_id')['cvd_hard'].nunique().max() == 1, "cvd_hard varies within participant"
assert df.groupby('participant_id')['cvd_composite'].nunique().max() == 1, "cvd_composite varies within participant"

# Participant-level prevalence
outcome = df.groupby('participant_id')[['cvd_hard', 'cvd_composite']].first()
print(f"Hard CVD  (primary)    : {outcome['cvd_hard'].mean():.1%}  (n={outcome['cvd_hard'].sum():,})")
print(f"Composite (sensitivity): {outcome['cvd_composite'].mean():.1%}  (n={outcome['cvd_composite'].sum():,})")

Hard CVD  (primary)    : 7.1%  (n=439)
Composite (sensitivity): 42.6%  (n=2,643)


## 3. Aggregate Accelerometer Data

Each participant has multiple rows — one per day of wear. This section:
1. **Quality-filters days**: keeps only days flagged as good quality (`accel_quality_flag == 0`) with ≥ 10 hours of wake wear time (≥ 600 min) — standard NHANES accelerometry criterion (Troiano et al. *Med Sci Sports Exerc* 2008)
2. **Requires ≥ 4 valid days**: minimum needed for reliable habitual activity estimates
3. **Aggregates to participant level**: mean activity levels and day-to-day variability across valid days

In [5]:
accel_agg = pd.read_pickle('data/processed/accel_agg.pkl')

print(f"Participants with accelerometer data (≥4 valid days): {len(accel_agg):,}")
print(f"\nValid days distribution:")
print(accel_agg['valid_days'].describe().apply(lambda x: f'{x:.1f}').to_string())

Participants with accelerometer data (≥4 valid days): 0

Valid days distribution:
count    0.0
mean     nan
std      nan
min      nan
25%      nan
50%      nan
75%      nan
max      nan


## 4. Biological Features

Biomarker and demographic data are constant per participant (duplicated across accelerometer rows). `build_biological_features()` takes the first row per participant and engineers derived clinical features.

**Derived features (`src/features.py`):**

| Feature | Formula | Rationale |
|---------|---------|-----------|
| `chol_hdl_ratio` | Total cholesterol / HDL | Stronger predictor of CVD than LDL alone; used in Framingham risk score |
| `non_hdl_cholesterol` | Total cholesterol − HDL | Captures all atherogenic lipoproteins (LDL + VLDL + IDL); preferred by ACC/AHA when triglycerides are elevated |
| `pulse_pressure` | Systolic BP − Diastolic BP | Marker of arterial stiffness; independent CVD predictor in older adults |
| `ever_smoker` | `smoked_100_cigarettes_lifetime == 1` | Standard epidemiological definition |
| `current_smoker` | `currently_smoke_cigarettes == 1` | Active smoking risk |

In [6]:
bio = build_biological_features(df)
print(f"Biological feature table: {bio.shape[0]:,} participants × {bio.shape[1]} columns")

Biological feature table: 6,205 participants × 20 columns


## 5. Merge into Analysis-Ready Dataset

Inner join: keeps only participants with both valid biomarker data **and** ≥ 4 valid accelerometer days. The join will reduce sample size — that's expected and correct.

> **Note on sample size:** participants without sufficient accelerometer data are excluded. This is the price of the digital biomarker analysis. The biological-only model in Section 6 can be evaluated on the full fasting subsample if needed — the feature set definitions intentionally keep the two analyses separable.

In [7]:
# Participant-level outcomes
outcome = df.groupby('participant_id')[['cvd_hard', 'cvd_composite']].first().reset_index()

# Merge: bio + accelerometer + outcome
analysis_df = (
    bio
    .merge(accel_agg, on='participant_id', how='inner')
    .merge(outcome,   on='participant_id', how='inner')
)

print(f"Participants in bio table    : {len(bio):,}")
print(f"Participants with accel data : {len(accel_agg):,}")
print(f"Final analysis dataset       : {len(analysis_df):,} participants × {analysis_df.shape[1]} cols")
print(f"\nHard CVD prevalence (final)  : {analysis_df['cvd_hard'].mean():.1%}")

Participants in bio table    : 6,205
Participants with accel data : 0
Final analysis dataset       : 0 participants × 30 cols

Hard CVD prevalence (final)  : nan%


## 6. Feature Sets

Three nested feature sets for modelling. The biological set can run on the full fasting subsample; the digital and combined sets are restricted to participants with valid accelerometer data.

**Design notes:**
- `ldl_mg_dl` is retained alongside `total_cholesterol_mg_dl` despite high collinearity (r=0.924) — tree models handle this, and both appear in different clinical risk scores. SHAP analysis in notebook 04 will reveal which the model actually uses.
- `hba1c_pct` is preferred over `fasting_glucose_mg_dl` as the primary glycaemia feature: it reflects 3-month average glucose and is less sensitive to fasting duration variability.
- `waist_circumference_cm` is retained over BMI as the primary adiposity measure: waist better captures visceral fat, which is more directly linked to CVD risk.

In [8]:
# Feature sets and targets are imported from src/features.py
# Leakage check
for col in TARGETS:
    for fset in [BIOLOGICAL_FEATURES, DIGITAL_FEATURES, COMBINED_FEATURES]:
        assert col not in fset, f"Leakage: {col} in feature set"

print(f"Biological : {len(BIOLOGICAL_FEATURES)} features")
print(f"Digital    : {len(DIGITAL_FEATURES)} features")
print(f"Combined   : {len(COMBINED_FEATURES)} features")

missing_cols = [c for c in COMBINED_FEATURES if c not in analysis_df.columns]
if missing_cols:
    print(f"\nWARNING — features not found in dataframe: {missing_cols}")
else:
    print("\nAll features present in analysis dataset.")

Biological : 19 features
Digital    : 8 features
Combined   : 27 features

All features present in analysis dataset.


## 7. Missingness & Imputation

Audit remaining missingness in the combined feature set, then apply median imputation. Median imputation is appropriate for tree-based models (LightGBM, TabPFN handle missingness natively, but a clean matrix simplifies pipeline comparisons).

In [9]:
missing = (
    analysis_df[COMBINED_FEATURES].isnull().mean()
    .sort_values(ascending=False)
    .rename('pct_missing')
)
missing_any = missing[missing > 0]

if missing_any.empty:
    print("No missing values in combined feature set.")
else:
    print(f"Features with missing values ({len(missing_any)}):")
    print(missing_any.apply(lambda x: f'{x:.2%}').to_string())

# Median imputation — fit on full dataset (train/test split happens in notebook 03)
imputation_medians = analysis_df[COMBINED_FEATURES].median()
analysis_df[COMBINED_FEATURES] = analysis_df[COMBINED_FEATURES].fillna(imputation_medians)

remaining_missing = analysis_df[COMBINED_FEATURES].isnull().sum().sum()
print(f"\nMissing values after imputation: {remaining_missing}")

No missing values in combined feature set.

Missing values after imputation: 0


## 8. Save

Save the analysis-ready dataset and feature set definitions. The train/test split is deferred to notebook 03 so that cross-validation folds are applied consistently across all models.

> **Imputation note:** medians were computed on the full dataset here. In notebook 03, re-fit imputation within each cross-validation fold to avoid data leakage.

In [10]:
os.makedirs('data/processed', exist_ok=True)

# Analysis-ready dataset
analysis_df.to_pickle('data/processed/analysis_ready.pkl')

# Feature set definitions + imputation medians
feature_sets = {
    'biological': BIOLOGICAL_FEATURES,
    'digital':    DIGITAL_FEATURES,
    'combined':   COMBINED_FEATURES,
    'targets':    TARGETS,
    'imputation_medians': imputation_medians.to_dict(),
}
with open('data/processed/feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)

print(f"Saved: data/processed/analysis_ready.pkl   {analysis_df.shape[0]:,} rows × {analysis_df.shape[1]} cols")
print(f"Saved: data/processed/feature_sets.json")
print(f"\nSummary:")
print(f"  Participants    : {len(analysis_df):,}")
print(f"  Hard CVD cases  : {analysis_df['cvd_hard'].sum():,}  ({analysis_df['cvd_hard'].mean():.1%})")
print(f"  Features (bio)  : {len(BIOLOGICAL_FEATURES)}")
print(f"  Features (dig)  : {len(DIGITAL_FEATURES)}")
print(f"  Features (all)  : {len(COMBINED_FEATURES)}")

Saved: data/processed/analysis_ready.pkl   0 rows × 30 cols
Saved: data/processed/feature_sets.json

Summary:
  Participants    : 0
  Hard CVD cases  : 0  (nan%)
  Features (bio)  : 19
  Features (dig)  : 8
  Features (all)  : 27
